In [25]:
import pandas as pd
from shared_modeling import load_or_create_master_split_ids, run_model_experiment

In [26]:
predictors = ['oDM', 'acog_PEgHTN', 'ChronHTN'] # diabetes, preeclampsia, chronic hypertension
df_health = pd.read_csv('../Data/PREGNANCY_OUTCOMES.csv', usecols=predictors + ['PublicID'])
df_outcomes = pd.read_csv('../Data/Modified/Outcome.csv', usecols=['PublicID', 'MH_outcome'])

# Create the master split once and persist it for reuse in other notebooks.
split_path = 'master_split_ids.csv'
train_ids, test_ids = load_or_create_master_split_ids(df_outcomes, split_path)
df_outcomes

,PublicID,MH_outcome
0,00004O,1
1,00007I,1
2,00008G,0
3,00015J,0
4,00016H,1
...,...,...
7736,17349I,1
7737,17350A,1
7738,17351V,0
7739,17352T,1


In [27]:

df = pd.merge(df_health, df_outcomes, on='PublicID', how='inner')
df


,PublicID,oDM,ChronHTN,acog_PEgHTN,MH_outcome
0,00004O,3.0,2.0,7.0,1
1,00007I,3.0,2.0,7.0,1
2,00008G,3.0,2.0,7.0,0
3,00015J,3.0,2.0,7.0,0
4,00016H,3.0,2.0,7.0,1
...,...,...,...,...,...
7736,17349I,3.0,2.0,7.0,1
7737,17350A,3.0,2.0,3.0,1
7738,17351V,3.0,2.0,7.0,0
7739,17352T,3.0,2.0,7.0,1


In [28]:
X = df.drop(['MH_outcome', 'PublicID'], axis=1)
y = df['MH_outcome']

train_df = df[df['PublicID'].isin(train_ids)].copy()
test_df = df[df['PublicID'].isin(test_ids)].copy()

X_train = train_df.drop(['MH_outcome', 'PublicID'], axis=1)
X_test = test_df.drop(['MH_outcome', 'PublicID'], axis=1)
y_train = train_df['MH_outcome']
y_test = test_df['MH_outcome']

y.value_counts()


MH_outcome
0    4591
1    3150
Name: count, dtype: int64

In [29]:
X

,oDM,ChronHTN,acog_PEgHTN
0,3.0,2.0,7.0
1,3.0,2.0,7.0
2,3.0,2.0,7.0
3,3.0,2.0,7.0
4,3.0,2.0,7.0
...,...,...,...
7736,3.0,2.0,7.0
7737,3.0,2.0,3.0
7738,3.0,2.0,7.0
7739,3.0,2.0,7.0


In [30]:
# Run the LR pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'lr',
    [],
    predictors
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters found: {'classifier__C': 0.001, 'classifier__l1_ratio': 0.0}
Best Macro F1 Score: 0.4884
Model Coefficients:
cat__oDM_1.0: 0.0044497369950241665
cat__oDM_2.0: -0.0004954733022038206
cat__oDM_3.0: -0.003681625406462689
cat__acog_PEgHTN_1.0: 0.0004969938331785703
cat__acog_PEgHTN_2.0: 0.0033220832266785735
cat__acog_PEgHTN_3.0: 0.01078848152366243
cat__acog_PEgHTN_4.0: -0.001303593221796096
cat__acog_PEgHTN_5.0: 0.016618162856123008
cat__acog_PEgHTN_6.0: -0.012338373820135898
cat__acog_PEgHTN_7.0: -0.01731111611135364
cat__ChronHTN_1.0: 0.02345919040208431
cat__ChronHTN_2.0: -0.02318655211572743
Evaluation Metrics for LR with shared preprocessing and macro F1 grid search:
Accuracy: 0.5662
Precision: 0.5122
Recall: 0.5076
F1-score: 0.4792
ROC AUC: 0.5086


In [31]:
# Run the RF pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'rf',
    [],
    predictors
)

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters found: {'classifier__max_depth': 20, 'classifier__min_samples_leaf': 4, 'classifier__min_samples_split': 3, 'classifier__n_estimators': 700}
Best Macro F1 Score: 0.4749
Feature Importances:
cat__oDM_1.0: 0.110362802686978
cat__oDM_2.0: 0.07973542460080194
cat__oDM_3.0: 0.09616643586839026
cat__acog_PEgHTN_1.0: 0.0
cat__acog_PEgHTN_2.0: 0.03352165282381018
cat__acog_PEgHTN_3.0: 0.07027053527164476
cat__acog_PEgHTN_4.0: 0.041223017879032584
cat__acog_PEgHTN_5.0: 0.09622965899303701
cat__acog_PEgHTN_6.0: 0.0714746086649096
cat__acog_PEgHTN_7.0: 0.08358637210146543
cat__ChronHTN_1.0: 0.1625710418617184
cat__ChronHTN_2.0: 0.15485844924821177
Evaluation Metrics for RF with shared preprocessing and macro F1 grid search:
Accuracy: 0.5617
Precision: 0.5091
Recall: 0.5060
F1-score: 0.4818
ROC AUC: 0.5056


In [32]:
# Run the XGBoost pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'xgb',
    [],
    predictors
)

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters found: {'classifier__colsample_bytree': 1.0, 'classifier__learning_rate': 0.01, 'classifier__max_depth': 7, 'classifier__n_estimators': 80, 'classifier__subsample': 0.7}
Best Macro F1 Score: 0.4822
Feature Importances:
cat__oDM_1.0: 0.050955094397068024
cat__oDM_2.0: 0.04350750893354416
cat__oDM_3.0: 0.09276965260505676
cat__acog_PEgHTN_1.0: 0.0
cat__acog_PEgHTN_2.0: 0.08185657858848572
cat__acog_PEgHTN_3.0: 0.07020751386880875
cat__acog_PEgHTN_4.0: 0.039801161736249924
cat__acog_PEgHTN_5.0: 0.1295967698097229
cat__acog_PEgHTN_6.0: 0.045469027012586594
cat__acog_PEgHTN_7.0: 0.06779740005731583
cat__ChronHTN_1.0: 0.37803930044174194
cat__ChronHTN_2.0: 0.0
Evaluation Metrics for XGB with shared preprocessing and macro F1 grid search:
Accuracy: 0.5591
Precision: 0.5040
Recall: 0.5026
F1-score: 0.4769
ROC AUC: 0.5056


In [33]:
# Run the SVM pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'svm',
    [],
    predictors
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters found: {'classifier__estimator__C': 0.1, 'classifier__estimator__gamma': 'auto', 'classifier__estimator__kernel': 'rbf'}
Best Macro F1 Score: 0.4948
Evaluation Metrics for SVM with shared preprocessing and macro F1 grid search:
Accuracy: 0.5449
Precision: 0.5072
Recall: 0.5061
F1-score: 0.4991
ROC AUC: 0.5087
